# 02 Train CNN-ResBiGRU

Ноутбук для обучения модели на подготовленных окнах временных рядов landmarks.

Основные классы: ResistanceBand, PushUps, Squats, RunInPlace.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from pathlib import Path

MODEL_PATH = Path("../experiments/best_model.pt")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ml.models.cnn_resbigru import CnnResBiGRU

num_features = 99  # 33 landmarks * (x, y, z)
num_classes = 4
model = CnnResBiGRU(in_features=num_features, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model.__class__.__name__, "ready")

In [ ]:
# synthetic smoke run (replace with real prepared windows)
X = torch.randn(64, 13, 99)
y = torch.randint(0, 4, (64,))
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)

model.train()
for epoch in range(2):
    total_loss = 0.0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch={epoch+1} loss={total_loss/len(loader):.4f}")

torch.save(model.state_dict(), MODEL_PATH)
print("Saved:", MODEL_PATH)